<style>
    h2 {
        color: #00246B;
        font-weight: bold;
        text-align: center;
        background-color: #CADCFC;
    }
</style>
<h2>
    Preparation
</h2>

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta

api_key = open("api_key.txt", "r").read()
cities = {
    "London": (51.5074, -0.1278),
    "Kent": (51.2787, 0.5217),
    "Essex": (51.7810, 0.2382),
    "Hertfordshire": (51.7670, -0.2087),
    "Surrey": (51.3148, -0.5590),
    "Buckinghamshire": (51.9827, -0.9844),
    "East Sussex": (50.8930, 0.0553),
    "West Sussex": (50.9280, -0.4610)
}

<style>
    h2 {
        color: #00246B;
        font-weight: bold;
        text-align: center;
        background-color: #CADCFC;
    }
</style>
<h2>
    Fetch from url with given latitude, longitude, start and end time
</h2>

In [ ]:
def get_weather_data(api_key, lat, lon, start_timestamp, end_timestamp):
    url = f"https://history.openweathermap.org/data/2.5/history/city?lat={lat}&lon={lon}&type=hour&start={start_timestamp}&end={end_timestamp}&appid={api_key}"
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error: {response.status_code}")
        return None

<style>
    h2 {
        color: #00246B;
        font-weight: bold;
        text-align: center;
        background-color: #CADCFC;
    }
</style>
<h2>
    Saving data between start and end date for given cities as csv
</h2>

In [ ]:
def process_and_save_weather_data(api_key, cities, start_date, end_date):
    all_weather_data = []
    for city, (lat, lon) in cities.items():
        weather_data = []
        current_date = datetime.strptime(start_date, "%Y-%m-%d %H:%M")
        end_date_dt = datetime.strptime(end_date, "%Y-%m-%d %H:%M")
        while current_date <= end_date_dt:
            start_timestamp = int(current_date.timestamp())
            end_timestamp = int((current_date + timedelta(days=1) - timedelta(seconds=1)).timestamp())

            print(f"Fetching data for {city} on {current_date.strftime('%Y-%m-%d')}...")

            data = get_weather_data(api_key, lat, lon, start_timestamp, end_timestamp)

            if data:
                for entry in data.get('list', []):
                    timestamp = datetime.utcfromtimestamp(entry['dt']).strftime('%Y-%m-%d %H:%M:%S')
                    main = entry['main']
                    wind = entry.get('wind', {})
                    clouds = entry.get('clouds', {})
                    weather = entry.get('weather', [{}])[0]
                    rain = entry.get('rain', {})
                    snow = entry.get('snow', {})
                    weather_data.append({
                        'Location': city,
                        'Timestamp': timestamp,
                        'Temperature (K)': main.get('temp'),
                        'Feels Like (K)': main.get('feels_like'),
                        'Pressure (hPa)': main.get('pressure'),
                        'Humidity (%)': main.get('humidity'),
                        'Temp Min (K)': main.get('temp_min'),
                        'Temp Max (K)': main.get('temp_max'),
                        'Sea Level Pressure (hPa)': main.get('sea_level'),
                        'Ground Level Pressure (hPa)': main.get('grnd_level'),
                        'Wind Speed (m/s)': wind.get('speed'),
                        'Wind Direction (°)': wind.get('deg'),
                        'Cloudiness (%)': clouds.get('all'),
                        'Weather Main': weather.get('main'),
                        'Weather Description': weather.get('description'),
                        'Weather Icon': weather.get('icon'),
                        'Rain (1h, mm)': rain.get('1h', 0),
                        'Rain (3h, mm)': rain.get('3h', 0),
                        'Snow (1h, mm)': snow.get('1h', 0),
                        'Snow (3h, mm)': snow.get('3h', 0)
                    })
            current_date += timedelta(days=1)
        all_weather_data.extend(weather_data)
    df = pd.DataFrame(all_weather_data)
    filename = f"WeatherData_AllCities_{start_date.replace(':', '').replace(' ', '_')}_to_{end_date_dt.strftime('%Y-%m-%d_%H-%M')}.csv"
    df.to_csv(filename, index=False)
    print(f"Data saved to {filename}")

<style>
    h2 {
        color: #00246B;
        font-weight: bold;
        text-align: center;
        background-color: #CADCFC;
    }
</style>
<h2>
    Usage
</h2>

In [ ]:
start_date = "2023-12-30 00:00"
end_date = "2024-12-30 23:59"
process_and_save_weather_data(api_key, cities, start_date, end_date)